In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# 
save_dir = './aug'
word = ['0', '1'] 
img_size = 128

X = []
Y = []

print("Loading...")
for idx, label in enumerate(word):
    folder_path = os.path.join(save_dir, label)
    if not os.path.exists(folder_path): continue
    
    files = os.listdir(folder_path)
    for f in files:
        full_path = os.path.join(folder_path, f)
        img = cv2.imread(full_path)
        if img is not None:
            img = cv2.resize(img, (img_size, img_size))
            X.append(img)
            Y.append(idx) # 

X = np.array(X, dtype="float32") / 255.0
Y = np.array(Y)

# 
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f"Data Segmentation: Train {len(X_train)}, Val {len(X_val)}, Test {len(X_test)}")

Loading...
Data Segmentation: Train 190, Val 48, Test 60


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_binary_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(img_size, img_size, 3)),
        MaxPooling2D(2, 2),
        
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Dropout(0.25),
        
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid') 
    ])
    
    model.compile(optimizer='adam', 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

model = build_binary_model()
model.summary()

history = model.fit(X_train, y_train, 
                    epochs=20, 
                    batch_size=32, 
                    validation_data=(X_val, y_val))

C:\Users\Administrator\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     7,372,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,392,449 (28.20 MB)

 Trainable params: 7,392,449 (28.20 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 174ms/step - accuracy: 0.4842 - loss: 1.2785 - val_accuracy: 0.5000 - val_loss: 0.6683
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - accuracy: 0.6211 - loss: 0.6046 - val_accuracy: 0.6458 - val_loss: 0.5593
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - accuracy: 0.7842 - loss: 0.5032 - val_accuracy: 0.6250 - val_loss: 0.4572
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - accuracy: 0.8263 - loss: 0.3777 - val_accuracy: 0.9167 - val_loss: 0.3271
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step - accuracy: 0.9421 - loss: 0.2143 - val_accuracy: 1.0000 - val_loss: 0.1265
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.9842 - loss: 0.0841 - val_accuracy: 1.0000 - val_loss: 0.0642
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step - accuracy: 1.0000 - loss: 0.0236 - val_accuracy: 1.0000 - val_loss: 0.0084
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step - accuracy: 0.9947 - loss: 0.0169 - val_accuracy: 1.0000 - val_loss:

In [3]:
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

score = model.evaluate(X_test, y_test, verbose=0)
print(f"Test (Loss): {score[0]:.4f}")
print(f"Test (Accuracy): {score[1]*100:.2f}%")

#tf.keras.models.load_model("model_trained_data.h5", compile=False)
model.save("model_trained_data.h5")
#keras.saving.save_model("model_trained_data.h5")
print("model_trained_data.h5 Saved")

Test (Loss): 0.0003
Test (Accuracy): 100.00%
model_trained_data.h5 Saved
